# EAP — Transformer NER Evaluation
**Question:** Is the ML_NER 75.6% result genuinely bad, or is it a test-suite artifact?

This notebook runs three evaluations:
1. **Fixed test suite** — same 158 addresses used in the paper (apples-to-apples)
2. **Hard cases** — addresses the rule-based system struggles with (favors transformer)
3. **Side-by-side** — BASELINE vs ML_NER on every address, per-case breakdown

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q rapidfuzz transformers torch sentence-transformers faiss-cpu

In [ ]:
# ── Clone repo ────────────────────────────────────────────────────────────────
import os
if not os.path.exists('EAP'):
    !git clone https://github.com/RealMati/EAP.git
os.chdir('EAP')
!git log --oneline -5

In [ ]:
# ── Build both parsers ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')
from eap.parser import EthiopianAddressParser

print('Loading BASELINE parser (rule-based only)...')
baseline = EthiopianAddressParser(data_dir='.', use_transformer_ner=False)
baseline.load()
print()

print('Loading ML_NER parser (rule-based + XLM-RoBERTa)...')
ml_ner = EthiopianAddressParser(
    data_dir='.', 
    use_transformer_ner=True,
    transformer_model='mbeukman/xlm-roberta-base-finetuned-ner-amharic'
)
ml_ner.load()
print('Both parsers ready.')

In [ ]:
# ── Helper: run a test list and print comparison ──────────────────────────────
def compare(tests, label=''):
    """
    tests: list of (address, expected_subcity, expected_landmark)
    expected values are substrings — 'unknown' means no expectation.
    """
    b_lm_ok = b_sc_ok = ml_lm_ok = ml_sc_ok = 0
    total = len(tests)
    rows = []

    for addr, exp_sc, exp_lm in tests:
        b  = baseline.parse(addr)
        ml = ml_ner.parse(addr)

        def sc_match(r, exp):
            if exp.lower() == 'unknown': return not r.subcity
            return bool(r.subcity and exp.lower() in r.subcity.lower())

        def lm_match(r, exp):
            if exp.lower() == 'unknown': return True
            return bool(r.landmark_name and exp.lower() in r.landmark_name.lower())

        b_sc  = sc_match(b, exp_sc)
        b_lm  = lm_match(b, exp_lm)
        ml_sc = sc_match(ml, exp_sc)
        ml_lm = lm_match(ml, exp_lm)

        b_sc_ok  += b_sc
        b_lm_ok  += b_lm
        ml_sc_ok += ml_sc
        ml_lm_ok += ml_lm

        icon_b  = '✓' if b_lm  else '✗'
        icon_ml = '✓' if ml_lm else '✗'
        delta   = '←ML wins' if (ml_lm and not b_lm) else ('←BASE wins' if (b_lm and not ml_lm) else '')

        rows.append((
            addr[:50],
            exp_lm,
            b.landmark_name or '(none)',
            ml.landmark_name or '(none)',
            icon_b, icon_ml, delta
        ))

    print(f'\n{"="*80}')
    print(f'  {label}   ({total} addresses)')
    print(f'{"="*80}')
    print(f'  {"Address":<50} {"Exp":<16} {"BASE":<25} {"ML_NER":<25} {""}')
    print(f'  {"-"*50} {"-"*16} {"-"*25} {"-"*25}')
    for r in rows:
        print(f'  {r[0]:<50} {r[1]:<16} {r[2][:24]:<25} {r[3][:24]:<25} {r[5]}{r[6]}')

    print(f'\n  Subcity  — BASELINE: {100*b_sc_ok/total:.1f}%   ML_NER: {100*ml_sc_ok/total:.1f}%')
    print(f'  Landmark — BASELINE: {100*b_lm_ok/total:.1f}%   ML_NER: {100*ml_lm_ok/total:.1f}%')
    return {
        'base_lm':  round(100*b_lm_ok/total, 1),
        'ml_lm':    round(100*ml_lm_ok/total, 1),
        'base_sc':  round(100*b_sc_ok/total, 1),
        'ml_sc':    round(100*ml_sc_ok/total, 1),
    }

## 1. Fixed Test Suite (same 158 addresses used in the paper)

In [ ]:
# Load the same test files used in test_v2.py
def load_tests(path):
    tests = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 3:
                tests.append((parts[0], parts[1], parts[2]))
    return tests

all_results = {}
for name, path in [
    ('English Only',          'tests/addresses_english_only.txt'),
    ('Transliterated Amharic','tests/addresses_transliterated.txt'),
    ('Mixed',                 'tests/addresses_mixed.txt'),
    ('Pure Amharic',          'tests/addresses_amharic_only.txt'),
]:
    tests = load_tests(path)
    all_results[name] = compare(tests, label=name)

In [ ]:
# Summary table for fixed suite
print('\n' + '='*60)
print('  FIXED TEST SUITE SUMMARY')
print('='*60)
print(f'  {"Category":<30} {"BASE LM":>8} {"ML LM":>8} {"Δ":>6}')
print(f'  {"-"*30} {"-"*8} {"-"*8} {"-"*6}')
total_b = total_ml = 0
for name, r in all_results.items():
    delta = r['ml_lm'] - r['base_lm']
    sign = '+' if delta >= 0 else ''
    print(f'  {name:<30} {r["base_lm"]:>7.1f}% {r["ml_lm"]:>7.1f}%  {sign}{delta:.1f}%')
    total_b  += r['base_lm']
    total_ml += r['ml_lm']
avg_b  = total_b  / len(all_results)
avg_ml = total_ml / len(all_results)
print(f'  {"AVERAGE":<30} {avg_b:>7.1f}% {avg_ml:>7.1f}%  {"+" if avg_ml>=avg_b else ""}{avg_ml-avg_b:.1f}%')

## 2. Hard Cases — addresses the rule-based system struggles with

These are designed to **favour the transformer**: ambiguous text, no explicit subcity, complex Amharic, descriptive rather than nominal.

In [ ]:
hard_cases = [
    # --- Ambiguous / no explicit subcity hint ---
    ('near the national stadium',            'kirkos',      'stadium'),
    ('opposite the unity park',              'arada',       'unity'),
    ('next to the railway station',          'lideta',      'railway'),
    ('behind the main post office',          'arada',       'post'),
    ('close to the old airport',             'bole',        'airport'),

    # --- Amharic only, no transliteration clue ---
    ('ከስታድዬሙ አጠገብ',                          'kirkos',      'stadium'),
    ('ከሂልተን ሆቴል ፊት ለፊት',                     'kirkos',      'hilton'),
    ('የቦሌ አለም አቀፍ አውሮፕላን ጣቢያ አካባቢ',          'bole',        'airport'),
    ('ሜስቀል አደባባይ ፊት',                         'kirkos',      'meskel'),
    ('ከሜርካቶ ጀርባ',                             'addis ketema','merkato'),

    # --- Mixed with noise ---
    ('Bole sub city near the Edna cinema',   'bole',        'edna'),
    ('around Piassa area Arada sub-city',    'arada',       'piassa'),
    ('Yeka woreda 3 CMC road',               'yeka',        'cmc'),
    ('Kirkos kazanchis area near Jupiter',   'kirkos',      'jupiter'),
    ('beside Mexico square Yeka',            'yeka',        'mexico'),

    # --- Transliterated with phonetic variants ---
    ('megenagna ayer meda airport yeka',     'yeka',        'unknown'),
    ('qirqos bete kristiyan akababi',        'kirkos',      'giorgis'),
    ('merkato addis ketema sefer',           'addis ketema','merkato'),
    ('saris nifas silk lafto',               'nifas silk',  'unknown'),
    ('lideta taitu hotel wede mexiko',       'lideta',      'taitu'),

    # --- Very short / minimal ---
    ('Bole',                                 'bole',        'unknown'),
    ('Kirkos',                               'kirkos',      'unknown'),
    ('CMC',                                  'yeka',        'cmc'),
    ('Piassa',                               'arada',       'piassa'),
    ('Sheraton',                             'kirkos',      'sheraton'),

    # --- Addresses where transformer should shine: entity in complex sentence ---
    ('I live near the Hilton Hotel in Kirkos',         'kirkos', 'hilton'),
    ('my shop is behind Meskel Square',                'kirkos', 'meskel'),
    ('delivery to Friendship Mall, Bole subcity',      'bole',   'friendship'),
    ('the office is next to Bole International Airport','bole',  'airport'),
    ('drop off at Sheraton Addis near Taitu Hotel',    'kirkos', 'sheraton'),
]

hard_results = compare(hard_cases, label='HARD CASES (favouring transformer)')

## 3. Cases Where ML_NER Specifically Wins or Loses

Isolate the disagreements to understand what the transformer actually changes.

In [ ]:
# Load all test suite addresses and find where the two parsers disagree
all_tests = []
for path in [
    'tests/addresses_english_only.txt',
    'tests/addresses_transliterated.txt',
    'tests/addresses_mixed.txt',
    'tests/addresses_amharic_only.txt',
]:
    all_tests.extend(load_tests(path))

ml_wins   = []   # ML correct, baseline wrong
base_wins = []   # Baseline correct, ML wrong
both_fail = []   # Both wrong

for addr, exp_sc, exp_lm in all_tests:
    if exp_lm.lower() == 'unknown':
        continue
    b  = baseline.parse(addr)
    ml = ml_ner.parse(addr)
    b_ok  = bool(b.landmark_name  and exp_lm.lower() in b.landmark_name.lower())
    ml_ok = bool(ml.landmark_name and exp_lm.lower() in ml.landmark_name.lower())

    if ml_ok and not b_ok:
        ml_wins.append((addr, exp_lm, b.landmark_name, ml.landmark_name))
    elif b_ok and not ml_ok:
        base_wins.append((addr, exp_lm, b.landmark_name, ml.landmark_name))
    elif not b_ok and not ml_ok:
        both_fail.append((addr, exp_lm, b.landmark_name, ml.landmark_name))

print(f'\nDisagreement analysis ({len(all_tests)} addresses with landmark expectation):')
print(f'  ML_NER wins (ML correct, BASE wrong):   {len(ml_wins)}')
print(f'  BASE wins (BASE correct, ML wrong):     {len(base_wins)}')
print(f'  Both fail:                              {len(both_fail)}')

if ml_wins:
    print(f'\n  ── ML wins ──')
    for addr, exp, b_lm, ml_lm in ml_wins:
        print(f'    {addr[:50]:<50}  exp={exp:<15} base={b_lm[:20]:<22} ml={ml_lm[:20]}')

if base_wins:
    print(f'\n  ── Baseline wins (ML broke these) ──')
    for addr, exp, b_lm, ml_lm in base_wins:
        print(f'    {addr[:50]:<50}  exp={exp:<15} base={b_lm[:20]:<22} ml={ml_lm[:20]}')

## 4. What is the Transformer Actually Extracting?

Inspect the raw transformer output to understand what it's tagging.

In [ ]:
from transformers import pipeline as hf_pipeline

ner_pipe = hf_pipeline(
    'ner',
    model='mbeukman/xlm-roberta-base-finetuned-ner-amharic',
    aggregation_strategy='simple'
)

probes = [
    'Bole Medhanealem bete kristiyan',
    'beside CMC Michael church Yeka',
    'ቦሌ ሜድሃኒአለም ቤተክርስቲያን አካባቢ',
    'ኪርቆስ ሂልተን ሆቴል ፊት',
    'Sheraton Hotel Kirkos subcity',
    'near the national stadium',
    'my shop is behind Meskel Square',
    'ሜስቀል አደባባይ ፊት',
    'atgeb CMC mikael bete kristian akababi baria building',
    'Yeka CMC Michael ቤተክርስቲያን አካባቢ',
]

print('Raw transformer NER output:')
print('='*80)
for text in probes:
    entities = ner_pipe(text)
    locs = [e for e in entities if 'LOC' in e.get('entity_group','')]
    print(f'\nInput:    {text}')
    if locs:
        for e in locs:
            print(f'  LOC → "{e["word"]}"  (conf={e["score"]:.2f})')
    else:
        print('  (no LOC entities tagged)')

## 5. Final Verdict

In [ ]:
print('='*60)
print('  FINAL VERDICT')
print('='*60)
print(f'\n  Fixed test suite:')
print(f'    BASELINE landmark accuracy: {avg_b:.1f}%')
print(f'    ML_NER   landmark accuracy: {avg_ml:.1f}%')
print(f'    Delta: {avg_ml - avg_b:+.1f}%')
print(f'\n  Hard cases (favouring transformer):')
print(f'    BASELINE: {hard_results["base_lm"]:.1f}%')
print(f'    ML_NER:   {hard_results["ml_lm"]:.1f}%')
print(f'    Delta: {hard_results["ml_lm"] - hard_results["base_lm"]:+.1f}%')
print(f'\n  Cases where ML_NER specifically broke things: {len(base_wins)}')
print(f'  Cases where ML_NER specifically helped:       {len(ml_wins)}')
if len(base_wins) > len(ml_wins):
    print(f'\n  CONCLUSION: ML_NER genuinely degrades accuracy.')
    print(f'  It breaks {len(base_wins)} addresses that baseline handles correctly,')
    print(f'  while only fixing {len(ml_wins)}. The 75.6% is real, not test-suite bias.')
elif len(ml_wins) > len(base_wins):
    print(f'\n  CONCLUSION: ML_NER helps on hard cases but hurt on the fixed suite.')
    print(f'  Test-suite bias is a factor — the fixed suite favours rule-based patterns.')
else:
    print(f'\n  CONCLUSION: ML_NER is roughly equivalent. Difference is noise.')